# Object Detection — Reviewed Results Walkthrough

This repository contains two stages of the project:
1. the original **YOLOv8 closed-set** detector and its saved scene tests;
2. the current **Grounding DINO open-vocabulary** application.

The two stages solve related but different problems, so their outputs should not be presented as one directly comparable benchmark.


## 1. Preserve the recorded YOLOv8 scene tests


In [1]:
# These are the scene-level observations documented by the original YOLO experiment.
tests = [
    ("Street scene", 5, "95% bus"),
    ("Kitchen scene", 9, "96% person"),
    ("Sports scene", 4, "94% person"),
    ("Wildlife (fox)", 2, "69% cat"),
]
for scene, count, top in tests:
    print(f"{scene:18s} objects={count:2d}  top_confidence={top}")


Street scene       objects= 5  top_confidence=95% bus
Kitchen scene      objects= 9  top_confidence=96% person
Sports scene       objects= 4  top_confidence=94% person
Wildlife (fox)     objects= 2  top_confidence=69% cat


### What this tells us

The common COCO scenes produce confident detections, while the fox example exposes the closed-set limitation: the model is forced to map an unfamiliar object onto one of the 80 known COCO classes. A high confidence score therefore does **not** guarantee that the label is semantically correct outside the training vocabulary.


## 2. Document the out-of-distribution failure pattern


In [1]:
# The original tests recorded nearest-known-class substitutions for unseen animals.
ood_examples = {
    "fox": "cat + dog",
    "rabbit": "cat",
    "squirrel": "bear",
}
for actual, predicted in ood_examples.items():
    print(f"{actual:8s} -> {predicted}")


fox      -> cat + dog
rabbit   -> cat
squirrel -> bear


### What this tells us

These are useful failure cases because they motivated the project's next design decision. The problem is not merely threshold tuning; it is **vocabulary closure**. YOLOv8 cannot return a class it was never trained to predict.


## 3. Show what changed in the current application


In [1]:
# The current app replaces the fixed COCO vocabulary with text-conditioned detection.
print("current_model: IDEA-Research/grounding-dino-base")
print("parameters: 232M")
print("task: zero-shot open-vocabulary object detection")
print("default_threshold: 0.30")
print("quick_presets: 8")


current_model: IDEA-Research/grounding-dino-base
parameters: 232M
task: zero-shot open-vocabulary object detection
default_threshold: 0.30
quick_presets: 8


### What this tells us

The upgrade changes the product requirement from 'choose among 80 known classes' to 'detect objects described in text.' That directly addresses the limitation exposed by the wildlife tests. It is an architectural evolution, not evidence that Grounding DINO has been quantitatively proven better on the same benchmark—the repository does not currently contain that controlled comparison.


## Final interpretation

The strongest story here is the **problem-driven upgrade path**: test a closed-set detector, observe out-of-distribution failures, then move to an open-vocabulary model designed for that failure mode.
